In [ ]:
using Pkg; Pkg.activate(joinpath(@__DIR__, ".."))
using LinearAlgebra
using HiLQR

In [ ]:
"""
Exponential spring force profile
"""

function ϕ(x::Vector)::Real
    return x[2]
end

function get_soft_fn(ϕ::Function, σ::Real, ρ::Real)
    fn = x -> σ * exp(-ρ * ϕ(x))# * ϕ(x)
    return fn
end

function get_soft_flow(bb_flow::Function, fn_soft::Function)
    soft_bb_flow = (x,u) -> bb_flow(x,u) + [zeros(3); fn_soft(x)]
end

In [ ]:
"""
Thrusted ball with plastic impact
"""

function flight_flow(x::Vector, u::Vector)::Vector
    return [x[3:4]; 0.0; u[1] - 9.81]
end

function slide_flow(x::Vector, u::Vector)::Vector
    return x
end

function impact_reset(x::Vector)::Vector
    return [x[1]; 1e-9; x[3]; 0.0]
end

function impact_guard(x::Vector, u::Vector)::Float64
    return x[2]
end

function get_plastic_ball_system()::HybridSystem
    nx = 4
    nu = 1
    modes = Dict(
        :flight => HybridMode(flight_flow),
        :slide => HybridMode(slide_flow)
    )
    impact_transition = Transition(
        flight_flow, slide_flow, impact_reset, impact_guard, nx
    )
    add_transition!(
        modes[:flight], modes[:slide], impact_transition
    )
    transitions = Dict(:impact => impact_transition)
    return HybridSystem(modes, transitions, nx, nu)
end

In [ ]:
"""
Solver Setup
"""

# Bouncing ball with thrust model
system = get_plastic_ball_system()

# Problem parameters
N = 50
Δt = 0.04

# Soft normal force profile
σ = 0.0#1e-1
ρ = 1e-1
soft_fn = get_soft_fn(ϕ, σ, ρ)
flight_flow = system.modes[:flight].flow
soft_flow = get_soft_flow(flight_flow, soft_fn)

# Stage and terminal costs
Q = 1e-6 * diagm([1.0, 1.0, 0.0, 0.0])
R = 1e-4 * I
Qf = 1e0 * Q

stage(x, u) = x'*Q*x + u'*R*u
terminal(x) = x'*Qf*x

params = HiLQR.ProblemParameters(
    system, soft_flow, stage, terminal, stage, terminal, N, Δt
)

# Reference trajectory and initial conditions
xref = [10.0; 0.0; 0.0; 0.0]
uref = zeros(system.nu)
params.xrefs = [xref for k = 1:N]
params.urefs = [uref for k = 1:(N-1)]
params.x0 = [0.0, 4.0, 5.0, 0.0]
params.mI = :flight

In [ ]:
"""
Solve using HiLQR
"""

# Solve
sol = HiLQR.Solution(params)
sol.us = [10*ones(system.nu) for k = 1:(N-1)]
cache = HiLQR.SolverCache(params)
opts = HiLQR.SolverOptions(multishoot=false, max_step=1.0)
@time HiLQR.solve!(sol, cache, params, opts)

# Visualize states
plot_2d_states(sol.xs, (1,2))